# YOLOv5 vs YOLOv8 — Comparaison sur le Dataset Pascal VOC

Ce notebook compare les performances de **YOLOv5** et **YOLOv8** sur le dataset Pascal VOC (~400 MB).

**Plan :**
1. Téléchargement et extraction du dataset VOC
2. Organisation dans le répertoire `data/`
3. Conversion des annotations VOC → format YOLO
4. Création des fichiers de configuration
5. Installation des dépendances YOLOv5 & YOLOv8
6. Initialisation des modèles
7. Entraînement YOLOv5
8. Entraînement YOLOv8
9. Chargement des résultats
10. Graphiques de comparaison (mAP, Loss, Précision/Rappel, Vitesse)

## 1. Téléchargement et extraction du dataset Pascal VOC

In [ ]:
import os
import urllib.request
import tarfile
import zipfile
from pathlib import Path

# Répertoires de base
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
VOC_DIR = DATA_DIR / "VOCdevkit"
DATA_DIR.mkdir(exist_ok=True)

# URLs officielles Pascal VOC 2007 (~450 MB train+val+test)
VOC_URLS = {
    "trainval": "http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar",
    "test":     "http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtest_06-Nov-2007.tar",
}

def download_with_progress(url: str, dest: Path):
    """Télécharge un fichier avec affichage de la progression."""
    filename = dest / Path(url).name
    if filename.exists():
        print(f"[SKIP] {filename.name} déjà téléchargé.")
        return filename

    print(f"[DOWNLOAD] {url}")
    downloaded = [0]
    total = [0]

    def reporthook(block_num, block_size, total_size):
        downloaded[0] = block_num * block_size
        total[0] = total_size
        if total_size > 0:
            pct = min(downloaded[0] / total_size * 100, 100)
            print(f"\r  {pct:.1f}% ({downloaded[0] / 1e6:.1f} / {total_size / 1e6:.1f} MB)", end="")

    urllib.request.urlretrieve(url, filename, reporthook)
    print(f"\n[OK] Sauvegardé dans {filename}")
    return filename


for split, url in VOC_URLS.items():
    tar_path = download_with_progress(url, DATA_DIR)
    if not (DATA_DIR / "VOCdevkit").exists():
        print(f"[EXTRACT] {tar_path.name} ...")
        with tarfile.open(tar_path) as t:
            t.extractall(DATA_DIR)
        print("[OK] Extraction terminée.")

print("\n[OK] Dataset VOC 2007 prêt dans", VOC_DIR)

## 2. Organisation du dataset dans le répertoire `data/`

In [ ]:
import shutil
import xml.etree.ElementTree as ET

VOC2007_DIR = VOC_DIR / "VOC2007"
IMG_SRC     = VOC2007_DIR / "JPEGImages"
ANN_SRC     = VOC2007_DIR / "Annotations"
SETS_SRC    = VOC2007_DIR / "ImageSets" / "Main"

# Répertoires cibles
IMAGES_DIR = DATA_DIR / "images"
LABELS_DIR = DATA_DIR / "labels"

for split in ("train", "val", "test"):
    (IMAGES_DIR / split).mkdir(parents=True, exist_ok=True)
    (LABELS_DIR / split).mkdir(parents=True, exist_ok=True)

# Mapping split files VOC → nos splits
SPLIT_MAP = {
    "train": SETS_SRC / "trainval.txt",
    "val":   SETS_SRC / "val.txt",
    "test":  SETS_SRC / "test.txt",
}

split_ids = {}
for split, txt_file in SPLIT_MAP.items():
    if not txt_file.exists():
        # fallback: utiliser train.txt si val.txt absent
        alt = SETS_SRC / f"{split}.txt"
        txt_file = alt if alt.exists() else list(SETS_SRC.glob("*.txt"))[0]
    with open(txt_file) as f:
        ids = [line.strip().split()[0] for line in f if line.strip()]
    split_ids[split] = ids
    print(f"  {split}: {len(ids)} images")

# Copier les images dans les bons dossiers
for split, ids in split_ids.items():
    for img_id in ids:
        src = IMG_SRC / f"{img_id}.jpg"
        dst = IMAGES_DIR / split / f"{img_id}.jpg"
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)

print("[OK] Images organisées dans data/images/{train,val,test}/")

## 3. Conversion des annotations VOC (XML) → format YOLO

In [ ]:
import os
import xml.etree.ElementTree as ET
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

VOC_CLASSES = [
    "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor"
]
CLASS2IDX = {c: i for i, c in enumerate(VOC_CLASSES)}


def _convert_and_write(args):
    """Convertit un XML VOC → YOLO et écrit le .txt — exécuté en parallèle."""
    xml_path_str, label_path_str, class2idx = args
    xml_path   = Path(xml_path_str)
    label_path = Path(label_path_str)
    if not xml_path.exists() or label_path.exists():
        return 0
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        size = root.find("size")
        w = int(size.find("width").text)
        h = int(size.find("height").text)
        lines = []
        for obj in root.findall("object"):
            cls_name = obj.find("name").text.strip()
            if cls_name not in class2idx:
                continue
            diff = obj.find("difficult")
            if diff is not None and int(diff.text) == 1:
                continue
            cls_id = class2idx[cls_name]
            bb = obj.find("bndbox")
            xmin = float(bb.find("xmin").text)
            ymin = float(bb.find("ymin").text)
            xmax = float(bb.find("xmax").text)
            ymax = float(bb.find("ymax").text)
            lines.append(
                f"{cls_id} {(xmin+xmax)/2/w:.6f} {(ymin+ymax)/2/h:.6f} "
                f"{(xmax-xmin)/w:.6f} {(ymax-ymin)/h:.6f}"
            )
        label_path.write_text("\n".join(lines))
        return 1
    except Exception:
        return 0


# ── Construction de la liste de tâches ───────────────────────────────────────
tasks = [
    (str(ANN_SRC / f"{img_id}.xml"), str(LABELS_DIR / split / f"{img_id}.txt"), CLASS2IDX)
    for split, ids in split_ids.items()
    for img_id in ids
]

# ── Conversion parallèle — tâche one-shot, tous les vCPUs disponibles ────────
NUM_WORKERS = os.cpu_count()   # 16 vCPUs → tous utilisés (tâche courte)
print(f"[INFO] {len(tasks)} fichiers à convertir sur {NUM_WORKERS} workers ...")

total_converted, done = 0, 0
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = [executor.submit(_convert_and_write, t) for t in tasks]
    for fut in as_completed(futures):
        total_converted += fut.result()
        done += 1
        if done % 1000 == 0 or done == len(tasks):
            print(f"\r  {done}/{len(tasks)} ({done/len(tasks)*100:.0f}%)", end="", flush=True)

print(f"\n[OK] {total_converted} fichiers convertis en format YOLO.")

## 4. Création des fichiers de configuration `data.yaml`

In [ ]:
import yaml

ABS_DATA = DATA_DIR.resolve()

# Utiliser des chemins absolus pour la portabilité
voc_yaml_content = {
    "path": str(ABS_DATA),
    "train": str(ABS_DATA / "images" / "train"),
    "val":   str(ABS_DATA / "images" / "val"),
    "test":  str(ABS_DATA / "images" / "test"),
    "nc": len(VOC_CLASSES),
    "names": VOC_CLASSES,
}

# Fichier unique partagé par YOLOv5 et YOLOv8
YAML_PATH = DATA_DIR / "voc.yaml"
with open(YAML_PATH, "w") as f:
    yaml.dump(voc_yaml_content, f, default_flow_style=False, allow_unicode=True)

print(f"[OK] Fichier de configuration créé : {YAML_PATH}")
print(yaml.dump(voc_yaml_content, default_flow_style=False))

## 5. Installation des dépendances YOLOv5 et YOLOv8

In [ ]:
import subprocess, sys

# ── YOLOv8 via Ultralytics (pip) ──────────────────────────────────────────────
subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics", "-q"], check=True)
print("[OK] Ultralytics (YOLOv8) installé.")

# ── YOLOv5 — cloner le dépôt officiel si nécessaire ──────────────────────────
YOLOV5_DIR = BASE_DIR / "yolov5"
if not YOLOV5_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/ultralytics/yolov5.git", str(YOLOV5_DIR)],
        check=True,
    )
    print("[OK] Dépôt YOLOv5 cloné.")
else:
    print("[SKIP] Dépôt YOLOv5 déjà présent.")

# Installer les dépendances YOLOv5
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(YOLOV5_DIR / "requirements.txt"), "-q"],
    check=True,
)
print("[OK] Dépendances YOLOv5 installées.")

## 6. Initialisation du modèle YOLOv5

In [ ]:
import torch
import os

DEVICE = "0" if torch.cuda.is_available() else "cpu"
if DEVICE == "0":
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"[INFO] GPU    : {torch.cuda.get_device_name(0)}  ({vram_gb:.0f} GB VRAM)")
else:
    vram_gb = 0
    print("[INFO] Dispositif : CPU")
print(f"[INFO] vCPUs  : {os.cpu_count()}")

# ── Hyperparamètres optimisés pour A100 SXM 80 GB + 16 vCPUs ─────────────────
EPOCHS   = 100
BATCH    = 128   # ~15 GB VRAM pour yolov5s/yolov8n à 640px sur 80 GB dispo
IMG_SZ   = 640   # taille standard YOLO
WORKERS  = 8     # num_cpus // 2 — laisse des cœurs au process principal

# Chemins de sortie
YV5_RUN_DIR = BASE_DIR / "runs" / "yolov5"
YV8_RUN_DIR = BASE_DIR / "runs" / "yolov8"
YV5_RUN_DIR.mkdir(parents=True, exist_ok=True)
YV8_RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n[INFO] Config : epochs={EPOCHS} | batch={BATCH} | img={IMG_SZ} | workers={WORKERS}")
print(f"[INFO] VRAM estimée ≈ {BATCH * 0.12:.0f}–{BATCH * 0.15:.0f} GB / {vram_gb:.0f} GB disponible")
print(f"[INFO] Sorties → {YV5_RUN_DIR}  /  {YV8_RUN_DIR}")

## 7. Initialisation du modèle YOLOv8

In [ ]:
from ultralytics import YOLO

# Charger YOLOv8n pré-entraîné (nano — le plus rapide)
yolov8_model = YOLO("yolov8n.pt")
print(f"[INFO] YOLOv8n chargé — paramètres : {sum(p.numel() for p in yolov8_model.model.parameters()):,}")
print(f"[INFO] YOLOv8  — modèle : yolov8n  | epochs : {EPOCHS} | batch : {BATCH} | img : {IMG_SZ}")
print(f"[INFO] Répertoire de sorties YOLOv8 : {YV8_RUN_DIR}")

## 8. Entraînement YOLOv5 sur VOC

In [ ]:
import time
import subprocess
import sys
from pathlib import Path
from ultralytics import YOLO

# ── Fonction utilitaire commune ───────────────────────────────────────────────
def train_model(name: str, run_fn, result_dir: Path) -> Path:
    """Wrapper partagé : lance l'entraînement, affiche la progression en temps réel."""
    sep = "─" * 62
    print(f"\n{sep}")
    print(f"  🚀  Démarrage entraînement  {name}")
    print(f"{sep}")
    t0 = time.time()
    rc = run_fn()
    elapsed = time.time() - t0
    status = "✅ OK" if rc == 0 else f"❌ ERREUR (code {rc})"
    print(f"\n{sep}")
    print(f"  {status}  {name}  —  {elapsed / 60:.1f} min")
    print(f"  📁  Résultats : {result_dir}")
    print(f"{sep}\n")
    return result_dir


def _run_yolov5() -> int:
    """Lance train.py YOLOv5 et streame chaque ligne en temps réel."""
    cmd = [
        sys.executable, str(YOLOV5_DIR / "train.py"),
        "--img",      str(IMG_SZ),
        "--batch",    str(BATCH),
        "--epochs",   str(EPOCHS),
        "--data",     str(YAML_PATH.resolve()),
        "--weights",  "yolov5s.pt",
        "--project",  str(YV5_RUN_DIR),
        "--name",     "voc",
        "--exist-ok",
        "--device",   DEVICE,
        "--workers",  str(WORKERS),   # 16 workers DataLoader
    ]
    with subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        encoding="utf-8",
        errors="replace",
    ) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
    return proc.returncode


def _run_yolov8() -> int:
    """Lance model.train() YOLOv8 — la lib Ultralytics streame elle-même."""
    model = YOLO("yolov8n.pt")
    model.train(
        data     = str(YAML_PATH.resolve()),
        epochs   = EPOCHS,
        batch    = BATCH,
        imgsz    = IMG_SZ,
        device   = DEVICE,
        project  = str(YV8_RUN_DIR),
        name     = "voc",
        exist_ok = True,
        workers  = WORKERS,   # 16 workers DataLoader
        verbose  = True,
    )
    return 0


# ── Entraînement YOLOv5 ───────────────────────────────────────────────────────
YV5_RESULT_DIR = train_model("YOLOv5s", _run_yolov5, YV5_RUN_DIR / "voc")

## 9. Entraînement YOLOv8 sur VOC

In [ ]:
import time, subprocess, sys
from pathlib import Path
from ultralytics import YOLO

# Redéfinir train_model (kernel a redémarré, cellule 17 non relancée)
def train_model(name: str, run_fn, result_dir: Path) -> Path:
    sep = "─" * 62
    print(f"\n{sep}")
    print(f"  🚀  Démarrage entraînement  {name}")
    print(f"{sep}")
    t0 = time.time()
    rc = run_fn()
    elapsed = time.time() - t0
    status = "✅ OK" if rc == 0 else f"❌ ERREUR (code {rc})"
    print(f"\n{sep}")
    print(f"  {status}  {name}  —  {elapsed / 60:.1f} min")
    print(f"  📁  Résultats : {result_dir}")
    print(f"{sep}\n")
    return result_dir

# YOLOv5 est déjà entraîné — pointer directement vers ses résultats sur disque
YV5_RESULT_DIR = Path("runs/yolov5/voc")
print(f"[OK] YOLOv5 results : {YV5_RESULT_DIR}  —  existe : {YV5_RESULT_DIR.exists()}")

In [ ]:
# ── Redéfinir avec yolov8s (small) pour comparaison équitable avec yolov5s ───
def _run_yolov8() -> int:
    model = YOLO("yolov8s.pt")
    model.train(
        data     = str(YAML_PATH.resolve()),
        epochs   = EPOCHS,
        batch    = BATCH,
        imgsz    = IMG_SZ,
        device   = DEVICE,
        project  = str(YV8_RUN_DIR),
        name     = "voc",
        exist_ok = True,
        workers  = WORKERS,
        verbose  = True,
    )
    return 0

# ── Entraînement YOLOv8s ─────────────────────────────────────────────────────
YV8_RESULT_DIR = train_model("YOLOv8s", _run_yolov8, YV8_RUN_DIR / "voc")

## 10. Chargement des résultats d'entraînement

In [ ]:
import pandas as pd

# ── YOLOv5 results ────────────────────────────────────────────────────────────
v5_csv = YV5_RESULT_DIR / "results.csv"
df_v5 = pd.read_csv(v5_csv)
df_v5.columns = df_v5.columns.str.strip()   # enlever les espaces
print("Colonnes YOLOv5 :", list(df_v5.columns))

# ── YOLOv8 results ────────────────────────────────────────────────────────────
v8_csv = YV8_RESULT_DIR / "results.csv"
df_v8 = pd.read_csv(v8_csv)
df_v8.columns = df_v8.columns.str.strip()
print("Colonnes YOLOv8 :", list(df_v8.columns))

df_v5["model"] = "YOLOv5s"
df_v8["model"] = "YOLOv8s"

print(f"\n[OK] YOLOv5 : {len(df_v5)} epochs chargées.")
print(f"[OK] YOLOv8 : {len(df_v8)} epochs chargées.")

## 11. Comparaison mAP (mAP50 et mAP50-95)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Mapper les noms de colonnes selon la version YOLO ────────────────────────
# YOLOv5 : "metrics/mAP_0.5", "metrics/mAP_0.5:0.95"
# YOLOv8 : "metrics/mAP50(B)", "metrics/mAP50-95(B)"
def get_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"Aucune colonne trouvée parmi {candidates}\nDisponibles : {list(df.columns)}")

V5_MAP50    = get_col(df_v5, ["metrics/mAP_0.5",      "metrics/mAP50(B)"])
V5_MAP5095  = get_col(df_v5, ["metrics/mAP_0.5:0.95", "metrics/mAP50-95(B)"])
V8_MAP50    = get_col(df_v8, ["metrics/mAP50(B)",      "metrics/mAP_0.5"])
V8_MAP5095  = get_col(df_v8, ["metrics/mAP50-95(B)",   "metrics/mAP_0.5:0.95"])

epochs_v5 = range(1, len(df_v5) + 1)
epochs_v8 = range(1, len(df_v8) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Comparaison mAP — YOLOv5s vs YOLOv8s (Pascal VOC 2007)", fontsize=14, fontweight="bold")

# mAP50
ax = axes[0]
ax.plot(epochs_v5, df_v5[V5_MAP50],   label="YOLOv5s", color="#2196F3", linewidth=2)
ax.plot(epochs_v8, df_v8[V8_MAP50],   label="YOLOv8s", color="#FF5722", linewidth=2)
ax.set_title("mAP @ IoU 0.50")
ax.set_xlabel("Epoch")
ax.set_ylabel("mAP50")
ax.legend()
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

# mAP50-95
ax = axes[1]
ax.plot(epochs_v5, df_v5[V5_MAP5095], label="YOLOv5s", color="#2196F3", linewidth=2)
ax.plot(epochs_v8, df_v8[V8_MAP5095], label="YOLOv8s", color="#FF5722", linewidth=2)
ax.set_title("mAP @ IoU 0.50 : 0.95")
ax.set_xlabel("Epoch")
ax.set_ylabel("mAP50-95")
ax.legend()
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.tight_layout()
plt.savefig(BASE_DIR / "map_comparison.png", dpi=150)
plt.show()
print(f"[OK] Graphique sauvegardé : map_comparison.png")

## 12. Comparaison des courbes de Loss (Box / Cls / Obj)

In [ ]:
# Colonnes de loss — YOLOv5 et YOLOv8 ont des noms légèrement différents
LOSS_MAP = {
    "Box Loss (train)": {
        "v5": get_col(df_v5, ["train/box_loss", "train/box_om"]),
        "v8": get_col(df_v8, ["train/box_loss", "train/box_om"]),
    },
    "Cls Loss (train)": {
        "v5": get_col(df_v5, ["train/cls_loss", "train/cls_om"]),
        "v8": get_col(df_v8, ["train/cls_loss", "train/cls_om"]),
    },
    "Box Loss (val)": {
        "v5": get_col(df_v5, ["val/box_loss", "val/box_om"]),
        "v8": get_col(df_v8, ["val/box_loss", "val/box_om"]),
    },
    "Cls Loss (val)": {
        "v5": get_col(df_v5, ["val/cls_loss", "val/cls_om"]),
        "v8": get_col(df_v8, ["val/cls_loss", "val/cls_om"]),
    },
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Courbes de Loss — YOLOv5s vs YOLOv8s", fontsize=14, fontweight="bold")

for ax, (title, cols) in zip(axes.flat, LOSS_MAP.items()):
    ax.plot(epochs_v5, df_v5[cols["v5"]], label="YOLOv5s", color="#2196F3", linewidth=2)
    ax.plot(epochs_v8, df_v8[cols["v8"]], label="YOLOv8s", color="#FF5722", linewidth=2)
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(BASE_DIR / "loss_comparison.png", dpi=150)
plt.show()
print("[OK] Graphique sauvegardé : loss_comparison.png")

## 13. Comparaison Précision & Rappel

In [ ]:
V5_PREC = get_col(df_v5, ["metrics/precision", "metrics/precision(B)"])
V5_REC  = get_col(df_v5, ["metrics/recall",    "metrics/recall(B)"])
V8_PREC = get_col(df_v8, ["metrics/precision(B)", "metrics/precision"])
V8_REC  = get_col(df_v8, ["metrics/recall(B)",    "metrics/recall"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Précision & Rappel — YOLOv5s vs YOLOv8s", fontsize=14, fontweight="bold")

ax = axes[0]
ax.plot(epochs_v5, df_v5[V5_PREC], label="YOLOv5s", color="#2196F3", linewidth=2)
ax.plot(epochs_v8, df_v8[V8_PREC], label="YOLOv8s", color="#FF5722", linewidth=2)
ax.set_title("Précision (Precision)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Précision")
ax.legend()
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

ax = axes[1]
ax.plot(epochs_v5, df_v5[V5_REC], label="YOLOv5s", color="#2196F3", linewidth=2)
ax.plot(epochs_v8, df_v8[V8_REC], label="YOLOv8s", color="#FF5722", linewidth=2)
ax.set_title("Rappel (Recall)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Rappel")
ax.legend()
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.tight_layout()
plt.savefig(BASE_DIR / "precision_recall_comparison.png", dpi=150)
plt.show()
print("[OK] Graphique sauvegardé : precision_recall_comparison.png")

## 14. Comparaison de vitesse d'inférence (ms/image & FPS)

In [ ]:
import glob
import numpy as np
import cv2

# Récupérer un échantillon d'images de test
test_images = sorted(glob.glob(str(IMAGES_DIR / "test" / "*.jpg")))[:50]
if not test_images:
    test_images = sorted(glob.glob(str(IMAGES_DIR / "val" / "*.jpg")))[:50]
print(f"[INFO] Benchmark sur {len(test_images)} images.")

# ── YOLOv5 inference speed ────────────────────────────────────────────────────
v5_model_path = str(YV5_RESULT_DIR / "weights" / "best.pt")
yv5_infer = torch.hub.load(str(YOLOV5_DIR), "custom", path=v5_model_path, source="local", verbose=False)
yv5_infer.eval()

# Warm-up
for _ in range(3):
    yv5_infer(test_images[0])

times_v5 = []
for img_path in test_images:
    t_start = time.perf_counter()
    yv5_infer(img_path)
    times_v5.append((time.perf_counter() - t_start) * 1000)

ms_v5  = float(np.mean(times_v5))
fps_v5 = 1000.0 / ms_v5
print(f"[YOLOv5s] {ms_v5:.2f} ms/image — {fps_v5:.1f} FPS")

# ── YOLOv8 inference speed ────────────────────────────────────────────────────
v8_model_path = str(YV8_RESULT_DIR / "weights" / "best.pt")
yv8_infer = YOLO(v8_model_path)

# Warm-up
for _ in range(3):
    yv8_infer(test_images[0], verbose=False)

times_v8 = []
for img_path in test_images:
    t_start = time.perf_counter()
    yv8_infer(img_path, verbose=False)
    times_v8.append((time.perf_counter() - t_start) * 1000)

ms_v8  = float(np.mean(times_v8))
fps_v8 = 1000.0 / ms_v8
print(f"[YOLOv8s] {ms_v8:.2f} ms/image — {fps_v8:.1f} FPS")

In [ ]:
import matplotlib.patches as mpatches

models = ["YOLOv5s", "YOLOv8s"]
ms_vals  = [ms_v5,  ms_v8]
fps_vals = [fps_v5, fps_v8]
colors   = ["#2196F3", "#FF5722"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Comparaison de Vitesse d'Inférence — YOLOv5s vs YOLOv8s", fontsize=14, fontweight="bold")

ax = axes[0]
bars = ax.bar(models, ms_vals, color=colors, width=0.4)
ax.set_title("Temps d'inférence (ms / image)")
ax.set_ylabel("ms")
ax.bar_label(bars, fmt="%.1f ms", padding=3, fontsize=11)
ax.set_ylim(0, max(ms_vals) * 1.3)
ax.grid(axis="y", alpha=0.3)

ax = axes[1]
bars = ax.bar(models, fps_vals, color=colors, width=0.4)
ax.set_title("FPS (images par seconde)")
ax.set_ylabel("FPS")
ax.bar_label(bars, fmt="%.1f FPS", padding=3, fontsize=11)
ax.set_ylim(0, max(fps_vals) * 1.3)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(BASE_DIR / "speed_comparison.png", dpi=150)
plt.show()
print("[OK] Graphique sauvegardé : speed_comparison.png")

## 15. Tableau récapitulatif final

In [ ]:
summary = pd.DataFrame({
    "Modèle":        ["YOLOv5s", "YOLOv8s"],
    "mAP50 (final)": [
        df_v5[V5_MAP50].iloc[-1],
        df_v8[V8_MAP50].iloc[-1],
    ],
    "mAP50-95 (final)": [
        df_v5[V5_MAP5095].iloc[-1],
        df_v8[V8_MAP5095].iloc[-1],
    ],
    "Précision (finale)": [
        df_v5[V5_PREC].iloc[-1],
        df_v8[V8_PREC].iloc[-1],
    ],
    "Rappel (final)": [
        df_v5[V5_REC].iloc[-1],
        df_v8[V8_REC].iloc[-1],
    ],
    "Inférence (ms)":  [ms_v5,  ms_v8],
    "FPS":             [fps_v5, fps_v8],
})

for col in ["mAP50 (final)", "mAP50-95 (final)", "Précision (finale)", "Rappel (final)"]:
    summary[col] = summary[col].map(lambda x: f"{x*100:.2f}%")
summary["Inférence (ms)"] = summary["Inférence (ms)"].map(lambda x: f"{x:.2f}")
summary["FPS"] = summary["FPS"].map(lambda x: f"{x:.1f}")

print("=" * 70)
print("  RÉSULTATS COMPARATIFS — YOLOv5s vs YOLOv8s sur Pascal VOC 2007")
print("=" * 70)
display(summary.set_index("Modèle"))